In [ ]:
# Cell 1: Installs

import subprocess, sys
pkgs = [
    "datasets", "transformers", "accelerate",
    "evaluate", "scikit-learn",
    "audiomentations", "librosa", "soundfile",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)


In [ ]:
# Cell 2: Imports

import os, gc, json, warnings, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from collections import Counter
from datasets import load_dataset, Audio, Dataset, DatasetDict
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from audiomentations import (
    Compose, AddGaussianNoise, PitchShift, TimeStretch,
    Shift, Gain, AddGaussianSNR
)

warnings.filterwarnings("ignore")

In [ ]:
# Cell 3: Constants

SR               = 16_000
CLIP_DURATION_S  = 1.0            # fillers are about 0.3-0.7s, 1s gives enough context
CLIP_SAMPLES     = int(SR * CLIP_DURATION_S)
MODEL_CHECKPOINT = "facebook/wav2vec2-base"
DATASET_ID       = "bookbot/PodcastFillers-filtered"
SAVE_DIR         = "filler_detector"
SEED             = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Cell 4: Load dataset

print(f"loading {DATASET_ID} ...")
raw = load_dataset(DATASET_ID)
print(raw)
print("columns:", raw["train"].column_names)

In [ ]:
# Cell 5: Explore what tokens exist in the timestamps


all_tokens = []
for split in raw:
    for sample in raw[split]:
        for ts in sample["timestamps"]:
            all_tokens.append(ts["text"].strip().lower())

token_counts = Counter(all_tokens)
print(f"total tokens: {len(all_tokens):,}")
print(f"unique: {len(token_counts):,}")
print(f"\ntop 80:")
for tok, cnt in token_counts.most_common(80):
    print(f"  {cnt:6d}  '{tok}'")

In [ ]:
# Cell 6: Define filler and skip token sets

FILLER_TOKENS = {
    'uh', 'um', 'uhm', 'umm', 'uhh', 'uum',
    'hmm', 'hm', 'mhm', 'mmhm', 'mm',
    'ah', 'er', 'err', 'eh',
    # leading-space variants common in this dataset
    ' um', ' uh', ' ah', ' er',
}

# these are not fillers and not clean speech either, just skip them
SKIP_TOKENS = {
    'breathing', 'laughing', 'music', 'noise',
    'applause', 'clapping', 'coughing', 'silence',
}

def classify_token(text):
    t = text.strip().lower()
    if t in FILLER_TOKENS or text.lower() in FILLER_TOKENS:
        return 'filler'
    if t in SKIP_TOKENS:
        return 'skip'
    return 'speech'

counts = Counter(classify_token(t) for t in all_tokens)
print(f"filler: {counts['filler']:,}")
print(f"skip:   {counts['skip']:,}")
print(f"speech: {counts['speech']:,}")

del all_tokens
gc.collect()

In [ ]:
# Cell 7: clip extraction + feature extraction


feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)

def extract_and_encode_split(dataset_split, split_name,
                              max_nonfiller_ratio=2.0,
                              clip_s=CLIP_DURATION_S):

    clip_samples = int(SR * clip_s)
    half_clip_ms = int(clip_s * 1000 / 2)

    # pass 1: find where the filler and speech tokens are (no audio loaded)
    filler_locs = []
    speech_locs = []

    for seg_idx, sample in enumerate(dataset_split):
        for ts_idx, ts in enumerate(sample["timestamps"]):
            cls = classify_token(ts["text"])
            if cls == 'filler':
                filler_locs.append((seg_idx, ts_idx))
            elif cls == 'speech':
                speech_locs.append((seg_idx, ts_idx))

    # downsample speech tokens so classes are roughly balanced
    max_speech = int(len(filler_locs) * max_nonfiller_ratio)
    if len(speech_locs) > max_speech:
        rng = np.random.RandomState(SEED)
        idx = rng.choice(len(speech_locs), size=max_speech, replace=False)
        speech_locs = [speech_locs[i] for i in sorted(idx)]

    # merge and sort by segment so we only decode each segment once
    all_locs = [(seg, ts, 1) for seg, ts in filler_locs] + \
               [(seg, ts, 0) for seg, ts in speech_locs]
    all_locs.sort(key=lambda x: x[0])

    print(f"  [{split_name}] {len(filler_locs):,} filler + "
          f"{len(speech_locs):,} speech = {len(all_locs):,} clips")

    # pass 2: load audio one segment at a time, cut clips, extract features
    all_features = []
    all_labels   = []

    split_audio = dataset_split.cast_column("audio", Audio(sampling_rate=SR))

    current_seg  = -1
    audio_array  = None
    timestamps   = None
    batch_clips  = []
    batch_labels = []
    BATCH_SIZE   = 128

    def flush_batch():
        nonlocal batch_clips, batch_labels
        if not batch_clips:
            return
        feats = feature_extractor(
            batch_clips, sampling_rate=SR, max_length=clip_samples,
            truncation=True, padding="max_length", return_tensors="np",
        )
        for row in feats["input_values"]:
            all_features.append(row)
        all_labels.extend(batch_labels)
        batch_clips  = []
        batch_labels = []

    processed = 0
    for seg_idx, ts_idx, label in all_locs:
        if seg_idx != current_seg:
            del audio_array, timestamps
            gc.collect()
            sample = split_audio[seg_idx]
            audio_array = np.array(sample["audio"]["array"], dtype=np.float32)
            timestamps  = sample["timestamps"]
            current_seg = seg_idx

        ts = timestamps[ts_idx]
        center_ms = (ts["start_time"] + ts["end_time"]) // 2

        clip_start_ms = max(0, center_ms - half_clip_ms)
        clip_end_ms   = clip_start_ms + int(clip_s * 1000)

        clip_start_s = int(clip_start_ms * SR / 1000)
        clip_end_s   = int(clip_end_ms * SR / 1000)

        total_s = len(audio_array)
        if clip_end_s > total_s:
            clip_end_s = total_s
            clip_start_s = max(0, clip_end_s - clip_samples)

        clip = audio_array[clip_start_s:clip_end_s]
        if len(clip) < clip_samples * 0.5:
            continue
        if len(clip) < clip_samples:
            clip = np.pad(clip, (0, clip_samples - len(clip)))

        peak = np.abs(clip).max()
        if peak < 1e-6:
            continue
        clip = clip / peak

        batch_clips.append(clip)
        batch_labels.append(label)

        if len(batch_clips) >= BATCH_SIZE:
            flush_batch()

        processed += 1
        if processed % 2000 == 0:
            print(f"    [{split_name}] {processed:,}/{len(all_locs):,}")

    flush_batch()
    del audio_array, timestamps, split_audio
    gc.collect()

    print(f"  [{split_name}] done: {len(all_labels):,} clips")

    # shuffle
    rng = np.random.RandomState(SEED)
    perm = rng.permutation(len(all_labels))
    all_features = [all_features[i] for i in perm]
    all_labels   = [all_labels[i] for i in perm]

    ds = Dataset.from_dict({"input_values": all_features, "labels": all_labels})
    ds.set_format("torch")

    del all_features
    gc.collect()

    return ds, all_labels

print("extraction function ready")

In [ ]:
# Cell 8: Run clip extraction on all splits

print("extracting clips...\n")

encoded_train, train_labels = extract_and_encode_split(
    raw["train"], "train", max_nonfiller_ratio=2.0)
gc.collect()

encoded_val, val_labels = extract_and_encode_split(
    raw["validation"], "val", max_nonfiller_ratio=3.0)
gc.collect()

encoded_test, test_labels = extract_and_encode_split(
    raw["test"], "test", max_nonfiller_ratio=3.0)
gc.collect()

# done with raw dataset
del raw
gc.collect()

print("\nsummary:")
for name, labels in [("train", train_labels), ("val", val_labels), ("test", test_labels)]:
    nf = sum(labels)
    nnf = len(labels) - nf
    print(f"  {name:6s}: {len(labels):6,d} clips  (filler: {nf:,d}  non-filler: {nnf:,d})")

print(f"\ninput shape: {tuple(encoded_train[0]['input_values'].shape)}")

In [ ]:
# Cell 9: Class weights and metrics

train_labels_np = np.array(train_labels, dtype=np.int64)
unique, counts = np.unique(train_labels_np, return_counts=True)
print("labels:", unique.tolist(), "counts:", counts.tolist())
assert len(unique) == 2, f"need 2 classes, got {len(unique)}"

weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_labels_np)
class_weights_tensor = torch.tensor(weights, dtype=torch.float32)
print(f"weights -- non-filler: {weights[0]:.4f}  filler: {weights[1]:.4f}")

acc_metric = evaluate.load("accuracy")
f1_metric  = evaluate.load("f1")

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    refs  = eval_pred.label_ids
    return {
        "accuracy": acc_metric.compute(predictions=preds, references=refs)["accuracy"],
        "f1_macro": f1_metric.compute(predictions=preds, references=refs, average="macro")["f1"],
    }

In [ ]:
# Cell 10: Collators

_aug = Compose([
    AddGaussianSNR(min_snr_db=15.0, max_snr_db=40.0, p=0.4),
    AddGaussianNoise(min_amplitude=0.0005, max_amplitude=0.008, p=0.3),
    PitchShift(min_semitones=-1.5, max_semitones=1.5, p=0.3),
    TimeStretch(min_rate=0.93, max_rate=1.07, p=0.25),
    Shift(min_shift=-0.08, max_shift=0.08, p=0.2),
    Gain(min_gain_db=-6, max_gain_db=6, p=0.3),
])

class AudioCollator:
    def __init__(self, training=False):
        self.training = training
    def __call__(self, features):
        wavs = np.stack([f["input_values"].numpy().astype(np.float32)
                         for f in features])
        if self.training:
            wavs = np.stack([_aug(w, sample_rate=SR) for w in wavs])
        return {
            "input_values": torch.tensor(wavs, dtype=torch.float32),
            "labels":       torch.tensor([f["labels"].item() for f in features],
                                          dtype=torch.long),
        }

train_collator = AudioCollator(training=True)
eval_collator  = AudioCollator(training=False)

In [ ]:
# Cell 11: Custom trainer with focal loss


class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ls = label_smoothing

    def forward(self, logits, targets):
        nc = logits.size(1)
        sm = torch.zeros_like(logits).fill_(self.ls / (nc - 1))
        sm.scatter_(1, targets.unsqueeze(1), 1.0 - self.ls)
        lp = F.log_softmax(logits, dim=1)
        pt = (torch.exp(lp) * sm).sum(1)
        fw = (1.0 - pt) ** self.gamma
        at = self.alpha.to(logits.device)[targets] if self.alpha is not None else 1.0
        return (at * fw * -(sm * lp).sum(1)).mean()


class FillerTrainer(Trainer):
    def __init__(self, *a, class_weights=None, eval_data_collator=None, **kw):
        super().__init__(*a, **kw)
        self._eval_dc = eval_data_collator
        self._fl = FocalLoss(alpha=class_weights, gamma=2.0)

    def get_eval_dataloader(self, eval_dataset=None):
        s = self.data_collator
        if self._eval_dc: self.data_collator = self._eval_dc
        dl = super().get_eval_dataloader(eval_dataset)
        self.data_collator = s
        return dl

    def get_test_dataloader(self, test_dataset):
        s = self.data_collator
        if self._eval_dc: self.data_collator = self._eval_dc
        dl = super().get_test_dataloader(test_dataset)
        self.data_collator = s
        return dl

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = self._fl(out.logits, labels)
        return (loss, out) if return_outputs else loss

In [ ]:
# Cell 12: Load model
# freeze the CNN feature extractor, let transformer layers train

model = AutoModelForAudioClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=2,
    label2id={"Non-filler": 0, "Filler": 1},
    id2label={0: "Non-filler", 1: "Filler"},
    ignore_mismatched_sizes=True,
)
model.config.mask_time_prob = 0.05
model.config.mask_time_length = 10
model.config.mask_feature_prob = 0.04
model.config.mask_feature_length = 10
model.wav2vec2.feature_extractor._freeze_parameters()

n = sum(p.numel() for p in model.parameters())
nf = sum(p.numel() for p in model.wav2vec2.feature_extractor.parameters())
print(f"total params: {n:,}  cnn frozen: {nf:,} ({100*nf/n:.1f}%)")

In [ ]:
# Cell 13: Phase 1 - head warmup (3 epochs)
# freeze entire backbone, only train classifier head


for p in model.wav2vec2.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = True
if hasattr(model, "projector"):
    for p in model.projector.parameters(): p.requires_grad = True

nt = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"phase 1 -- trainable: {nt:,} (head only)")

p1_args = TrainingArguments(
    output_dir="./fp1", eval_strategy="epoch", save_strategy="no",
    learning_rate=5e-4, per_device_train_batch_size=32,
    per_device_eval_batch_size=64, num_train_epochs=3,
    warmup_ratio=0.1, weight_decay=0.01, lr_scheduler_type="cosine",
    max_grad_norm=1.0, logging_steps=50, report_to="none",
    remove_unused_columns=False, fp16=torch.cuda.is_available(),
    dataloader_num_workers=2, seed=SEED,
)

t1 = FillerTrainer(
    model=model, args=p1_args,
    train_dataset=encoded_train, eval_dataset=encoded_val,
    compute_metrics=compute_metrics,
    data_collator=eval_collator, eval_data_collator=eval_collator,
    class_weights=class_weights_tensor,
)

print("\nstarting phase 1...")
t1.train()
r = t1.evaluate()
print(f"phase 1 val acc: {r.get('eval_accuracy','?'):.4f}  "
      f"f1: {r.get('eval_f1_macro','?'):.4f}")

In [ ]:
# Cell 14: Phase 2 -full fine-tuning (up to 15 epochs with early stopping)
# unfreeze transformer layers, keep CNN frozen

for p in model.wav2vec2.parameters(): p.requires_grad = True
model.wav2vec2.feature_extractor._freeze_parameters()

nt = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"phase 2 -- trainable: {nt:,}")

p2_args = TrainingArguments(
    output_dir="./fp2", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=1e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=64, gradient_accumulation_steps=2,
    num_train_epochs=15, warmup_ratio=0.05, weight_decay=0.01,
    lr_scheduler_type="cosine", max_grad_norm=1.0, logging_steps=50,
    report_to="none", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    remove_unused_columns=False, fp16=torch.cuda.is_available(),
    dataloader_num_workers=2, save_total_limit=2, seed=SEED,
)

t2 = FillerTrainer(
    model=model, args=p2_args,
    train_dataset=encoded_train, eval_dataset=encoded_val,
    compute_metrics=compute_metrics,
    data_collator=train_collator, eval_data_collator=eval_collator,
    class_weights=class_weights_tensor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

print("\nstarting phase 2...")
t2.train()
r = t2.evaluate()
print(f"phase 2 val acc: {r.get('eval_accuracy','?'):.4f}  "
      f"f1: {r.get('eval_f1_macro','?'):.4f}")

In [ ]:
# Cell 15: Check for overfitting
# compare train loss vs val loss from the training logs


train_log = t2.state.log_history

# pull per-epoch train loss and val loss
train_losses = []
val_losses   = []
val_accs     = []
val_f1s      = []

for entry in train_log:
    if "loss" in entry and "eval_loss" not in entry:
        train_losses.append(entry["loss"])
    if "eval_loss" in entry:
        val_losses.append(entry["eval_loss"])
        val_accs.append(entry.get("eval_accuracy", 0))
        val_f1s.append(entry.get("eval_f1_macro", 0))

# resample train losses to match epoch count (train logs more frequently)
n_epochs = len(val_losses)
if len(train_losses) > n_epochs:
    step = len(train_losses) / n_epochs
    epoch_train_losses = [train_losses[min(int((i+1)*step)-1, len(train_losses)-1)]
                          for i in range(n_epochs)]
else:
    epoch_train_losses = train_losses

print("per-epoch summary:")
print(f"{'epoch':>5}  {'train_loss':>10}  {'val_loss':>10}  {'val_acc':>8}  {'val_f1':>8}  {'gap':>8}")
print("-" * 60)
for i in range(n_epochs):
    tl = epoch_train_losses[i] if i < len(epoch_train_losses) else float('nan')
    vl = val_losses[i]
    gap = vl - tl
    print(f"{i+1:>5}  {tl:>10.4f}  {vl:>10.4f}  {val_accs[i]:>8.4f}  {val_f1s[i]:>8.4f}  {gap:>+8.4f}")

# plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, n_epochs + 1)
ax1.plot(epochs, epoch_train_losses[:n_epochs], 'o-', label='train loss')
ax1.plot(epochs, val_losses, 's-', label='val loss')
ax1.set_xlabel('epoch')
ax1.set_ylabel('loss')
ax1.set_title('train vs val loss (gap = overfitting)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, val_accs, 'o-', label='val accuracy')
ax2.plot(epochs, val_f1s, 's-', label='val f1')
ax2.axhline(0.85, color='red', ls='--', alpha=0.5, label='85% target')
ax2.set_xlabel('epoch')
ax2.set_ylabel('score')
ax2.set_title('validation accuracy and f1')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("overfitting_check.png", dpi=120)
plt.show()


best_val_acc = max(val_accs)
best_val_f1  = max(val_f1s)
final_gap    = val_losses[-1] - epoch_train_losses[min(len(epoch_train_losses)-1, n_epochs-1)]

print(f"\nbest val acc: {best_val_acc:.2%}")
print(f"best val f1:  {best_val_f1:.4f}")
print(f"final train-val loss gap: {final_gap:+.4f}")

if final_gap > 0.05:
    print("\n looks like some overfitting (val loss higher than train by >0.05)")
    print("    but if val accuracy is still 85%+ and test accuracy is close, its fine")
    print("    early stopping already picked the best checkpoint")
elif best_val_acc < 0.85:
    print("\n not overfitting but accuracy is below 85%")
    print("    try: increase epochs, lower learning rate, or check filler token set")
else:
    print("\n no overfitting")

In [ ]:
# Cell 16: Threshold tuning on validation set

vp = t2.predict(encoded_val)
vprobs = torch.softmax(torch.tensor(vp.predictions.astype(np.float32)), dim=1).numpy()
vacts  = vp.label_ids.astype(np.int64)

best_t, best_a = 0.5, 0.0
for t in np.arange(0.30, 0.70, 0.01):
    a = ((vprobs[:, 1] >= t).astype(int) == vacts).mean()
    if a > best_a: best_a, best_t = a, t

default_a = (np.argmax(vprobs, axis=1) == vacts).mean()
print(f"default 0.50 threshold: {default_a:.2%}")
print(f"optimal {best_t:.2f} threshold: {best_a:.2%}")

In [ ]:
# Cell 17: Final test evaluation
# this is the number that matters -- how well does it do on unseen data
# if test acc is close to val acc = no overfitting
# if test acc is way below val acc = overfitting

tp = t2.predict(encoded_test)
probs   = torch.softmax(torch.tensor(tp.predictions.astype(np.float32)), dim=1).numpy()
actuals = tp.label_ids.astype(np.int64)

acc_def  = float((np.argmax(probs, 1) == actuals).mean())
acc_tune = float(((probs[:, 1] >= best_t).astype(int) == actuals).mean())
accuracy = max(acc_def, acc_tune)
preds    = (probs[:, 1] >= best_t).astype(int) if acc_tune >= acc_def else np.argmax(probs, 1)

print(f"  val accuracy:           {best_a:.2%}")
print(f"  test accuracy (0.50):   {acc_def:.2%}")
print(f"  test accuracy ({best_t:.2f}):   {acc_tune:.2%}")
print(f"  val-test gap:           {abs(best_a - accuracy):.2%}")
print(f"")

if accuracy >= 0.85:
    print(f"  Passed. {accuracy:.2%} >= 85% target")
else:
    print(f"  Bellow target. {accuracy:.2%} < 85%")

if abs(best_a - accuracy) < 0.03:
    print(f"  No overfitting. val and test within 3%")
else:
    print(f"  Possible overfittng. val-test gap is {abs(best_a - accuracy):.2%}")

print("-"*55)

print("\n" + classification_report(actuals, preds,
                                    target_names=["Non-filler", "Filler"]))

# plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(confusion_matrix(actuals, preds),
    display_labels=["Non-filler", "Filler"]).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("confusion matrix")

fp = probs[:, 1]
axes[1].hist(fp[actuals==0], bins=50, alpha=0.7, label="non-filler")
axes[1].hist(fp[actuals==1], bins=50, alpha=0.7, label="filler")
axes[1].axvline(best_t, color='red', ls='--', label=f'threshold={best_t:.2f}')
axes[1].set_xlabel("P(filler)")
axes[1].set_title("probability distribution")
axes[1].legend()
plt.tight_layout()
plt.savefig("test_results.png", dpi=120)
plt.show()

In [ ]:
# Cell 18: Save model locally and zip for download

os.makedirs(SAVE_DIR, exist_ok=True)
t2.save_model(SAVE_DIR)
feature_extractor.save_pretrained(SAVE_DIR)

with open(os.path.join(SAVE_DIR, "config_extra.json"), "w") as f:
    json.dump({
        "optimal_threshold": float(best_t),
        "clip_duration_s": CLIP_DURATION_S,
        "sample_rate": SR,
        "filler_tokens": sorted(FILLER_TOKENS),
        "test_accuracy": float(accuracy),
        "val_accuracy": float(best_a),
    }, f, indent=2)

os.system(f"zip -r {SAVE_DIR}.zip {SAVE_DIR}")
print(f"saved to {SAVE_DIR}/")
print(f"download: {SAVE_DIR}.zip")

# auto-download in colab
try:
    from google.colab import files
    files.download(f"{SAVE_DIR}.zip")
except ImportError:
    pass

In [ ]:
# Cell 19: Inference helper functions


import librosa

inf_fe    = AutoFeatureExtractor.from_pretrained(SAVE_DIR)
inf_model = AutoModelForAudioClassification.from_pretrained(SAVE_DIR)
inf_model.eval()

with open(os.path.join(SAVE_DIR, "config_extra.json")) as f:
    _cfg = json.load(f)
    OPT_THRESH = _cfg["optimal_threshold"]

if torch.cuda.is_available(): inf_model = inf_model.cuda()
_dev = next(inf_model.parameters()).device


def predict_clip(audio, sr):
    """classify a ~1s audio clip as filler or non-filler"""
    a = np.array(audio, dtype=np.float32)
    if sr != SR: a = librosa.resample(a, orig_sr=sr, target_sr=SR)
    pk = np.abs(a).max()
    if pk > 0: a = a / pk
    f = inf_fe([a], sampling_rate=SR, max_length=CLIP_SAMPLES,
               truncation=True, padding="max_length", return_tensors="pt")
    f = {k: v.to(_dev) for k, v in f.items()}
    with torch.no_grad(): logits = inf_model(**f).logits
    p = torch.softmax(logits, -1)[0].cpu().numpy()
    idx = 1 if p[1] >= OPT_THRESH else 0
    return {
        "label": inf_model.config.id2label[idx],
        "confidence": round(float(p[idx]), 4),
        "filler_prob": round(float(p[1]), 4),
    }


def scan_audio(audio, sr, window_s=1.0, hop_s=0.3):
    """slide a window over long audio and return filler detections"""
    a = np.array(audio, dtype=np.float32)
    if sr != SR: a = librosa.resample(a, orig_sr=sr, target_sr=SR)
    pk = np.abs(a).max()
    if pk > 0: a = a / pk
    ws = int(window_s * SR)
    hs = int(hop_s * SR)
    results = []
    for s in range(0, len(a) - ws + 1, hs):
        r = predict_clip(a[s:s+ws], SR)
        r["start_s"] = s / SR
        r["end_s"] = (s + ws) / SR
        results.append(r)
    return results


print(f"threshold: {OPT_THRESH:.2f}")

In [ ]:
# Cell 20: Test on your own audio
# upload wav/mp3

try:
    from google.colab import files as cf
    import IPython.display as ipd

    print("upload a wav or mp3...")
    uploaded = cf.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        audio, sr = librosa.load(fname, sr=SR, mono=True)
        print(f"file: {fname}  duration: {len(audio)/sr:.2f}s")
        ipd.display(ipd.Audio(audio, rate=sr))

        results = scan_audio(audio, sr)

        print(f"\n{'#':<4} {'time':<14} {'pred':<14} {'p(filler)':>10}")
        print("-" * 48)
        for i, r in enumerate(results):
            m = " <-- filler" if r["label"] == "Filler" else ""
            print(f"{i+1:<4} {r['start_s']:.1f}s-{r['end_s']:.1f}s  "
                  f"{r['label']:<14} {r['filler_prob']:10.3f}{m}")

        ts = [(r['start_s']+r['end_s'])/2 for r in results]
        fp = [r['filler_prob'] for r in results]
        plt.figure(figsize=(12, 4))
        plt.plot(ts, fp, 'o-', ms=4)
        plt.axhline(OPT_THRESH, color='red', ls='--', label='threshold')
        plt.fill_between(ts, fp, OPT_THRESH,
                          where=[p >= OPT_THRESH for p in fp],
                          alpha=0.3, color='red', label='filler')
        plt.xlabel('time (s)'); plt.ylabel('p(filler)')
        plt.title(f"filler detection -- {fname}")
        plt.ylim(0, 1); plt.legend(); plt.tight_layout(); plt.show()

        nf = sum(1 for r in results if r['label']=='Filler')
        print(f"\nfiller windows: {nf}/{len(results)}")

except ImportError:
    print("not in colab  use scan_audio(audio_array, sr) or predict_clip(audio_array, sr)")